# Nixia GPU Training (Kaggle / Google Colab)

Notebook ini menjalankan build dataset, tokenizer, training GPU/CPU, eval, dan prompt eval.

Backend yang disarankan:
- Colab/Kaggle NVIDIA: `cuda` + Cargo feature `cuda-backend`.
- AMD/Intel lokal: `wgpu` + Cargo feature `wgpu-backend`.
- CPU fallback: `flex` tanpa feature GPU.


In [ ]:
# Runtime knobs. Adjust these before running all cells.
BACKEND = 'cuda'      # 'cuda', 'wgpu', or 'flex'
DEVICE_INDEX = 0
GPU_KIND = 'discrete' # wgpu only: default, discrete, integrated, virtual, cpu
BATCH_SIZE = 16       # increase until VRAM is almost full but stable
NUM_WORKERS = 2       # CPU dataloader workers; try 2-4 on hosted notebooks
EPOCHS = 20
SEQ_LEN = 128
LAYERS = 8
LR = '0.00005'
VOCAB_SIZE = 6000
ARTIFACTS = 'artifacts/nixia-gpu-run'
VOCAB = 'artifacts/vocab-gpu.txt'


In [ ]:
# Quiet setup helpers. Non-training setup/build commands only print a short status,
# and dump captured logs only when a command fails.
import json, os, pathlib, shlex, shutil, subprocess, textwrap

def run_quiet(cmd, label=None, timeout=None, env=None):
    label = label or cmd
    print(f'▶ {label}')
    result = subprocess.run(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
        env=env,
    )
    if result.returncode != 0:
        print(result.stdout[-6000:])
        raise subprocess.CalledProcessError(result.returncode, cmd)
    print(f'✓ {label}')
    return result

# Colab usually needs Rust. Kaggle often persists installs only for the session.
if shutil.which('cargo') is None:
    run_quiet(
        "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y",
        'Install Rust toolchain',
        timeout=900,
    )
    os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
run_quiet('rustc --version && cargo --version', 'Check Rust toolchain')


In [ ]:
# If this notebook is not already inside the repo, clone it here.
# Public default repo. For a private fork, set env NIXIA_REPO_URL first.
REPO_URL = os.environ.get('NIXIA_REPO_URL', 'https://github.com/eikarna/nixia-core.git')
REPO_DIR = 'nixia'
if not os.path.exists('Cargo.toml'):
    if os.path.exists(REPO_DIR) and not os.path.exists(os.path.join(REPO_DIR, 'Cargo.toml')):
        print(f'Removing incomplete clone at {REPO_DIR!r}')
        shutil.rmtree(REPO_DIR)
    if not os.path.exists(REPO_DIR):
        env = os.environ.copy()
        env['GIT_TERMINAL_PROMPT'] = '0'
        run_quiet(
            f'git clone --depth 1 {shlex.quote(REPO_URL)} {shlex.quote(REPO_DIR)}',
            f'Clone {REPO_URL}',
            timeout=600,
            env=env,
        )
    os.chdir(REPO_DIR)
print(f'Repo ready: {os.getcwd()}')
run_quiet('git status --short', 'Check git status')


In [ ]:
# GPU visibility check. NVIDIA should show nvidia-smi on Colab/Kaggle.
!nvidia-smi || true


In [ ]:
# Build and audit curated chat-clean corpus from checked-in templates.
run_quiet(
    "python tools/build_dataset.py --sources nixia_seed --max-rows-per-source 0 --synthesize 0 --valid-ratio 0.1 --min-score 0.8 --offline --extra-glob 'data/templates/nixia_dataset_*.txt' --output data/curated/chatclean_train.txt --valid-output data/curated/chatclean_valid.txt --report data/curated/chatclean_report.json",
    'Build curated dataset',
)
run_quiet(
    'python tools/audit_dataset.py --train data/curated/chatclean_train.txt --valid data/curated/chatclean_valid.txt --build-report data/curated/chatclean_report.json --json-output data/curated/chatclean_audit.json',
    'Audit curated dataset',
)
audit = json.loads(pathlib.Path('data/curated/chatclean_audit.json').read_text())
train = audit['splits']['train']['dialogues']
valid = audit['splits']['valid']['dialogues']
print(f"Dataset audit: status={audit['status']} readiness={audit['readiness']} train={train} valid={valid}")


In [ ]:
# Select Cargo features for the requested backend.
feature_map = {'cuda': 'cuda-backend', 'wgpu': 'wgpu-backend', 'flex': ''}
FEATURES = feature_map[BACKEND]
FEATURE_ARG = f'--features {FEATURES}' if FEATURES else ''
BACKEND_ARG = f'--backend {BACKEND}'
DEVICE_ARG = f'--device-index {DEVICE_INDEX}'
GPU_KIND_ARG = f'--gpu-kind {GPU_KIND}' if BACKEND == 'wgpu' else ''
BIN = './target/release/nixia'
print(f'Backend={BACKEND} features={FEATURES or "none"} device_index={DEVICE_INDEX}')


In [ ]:
# Build release binary quietly, then train tokenizer quietly.
# Training/eval cells below call the compiled binary directly, so Cargo build logs stay hidden.
run_quiet(f'cargo build --release {FEATURE_ARG}', 'Build release binary')
run_quiet(
    f'{BIN} tokenizer --corpus data/curated/chatclean_train.txt --vocab {VOCAB} --vocab-size {VOCAB_SIZE}',
    'Train tokenizer',
)


In [ ]:
# Long training. This cell intentionally keeps logs visible.
# For high GPU utilization, increase BATCH_SIZE gradually.
cmd = ' '.join(part for part in [
    BIN, 'train', BACKEND_ARG, GPU_KIND_ARG, DEVICE_ARG,
    '--preset nixia-micro', f'--seq-len {SEQ_LEN}', f'--layers {LAYERS}',
    '--corpus data/curated/chatclean_train.txt',
    '--valid data/curated/chatclean_valid.txt',
    f'--vocab {VOCAB}', f'--artifacts {ARTIFACTS}',
    f'--epochs {EPOCHS}', f'--batch-size {BATCH_SIZE}',
    f'--num-workers {NUM_WORKERS}', f'--lr {LR}',
] if part)
print(cmd)
subprocess.run(cmd, shell=True, check=True)


In [ ]:
# Evaluate validation loss/perplexity. This cell intentionally keeps output visible.
subprocess.run(
    f'{BIN} eval --corpus data/curated/chatclean_valid.txt --vocab {VOCAB} --artifacts {ARTIFACTS}',
    shell=True,
    check=True,
)


In [ ]:
# Prompt regression eval and one manual generation sample. Keep output visible.
subprocess.run(
    f'python tools/eval_prompts.py --artifacts {ARTIFACTS} --vocab {VOCAB} --output data/curated/prompt_eval_gpu.md',
    shell=True,
    check=True,
)
subprocess.run(
    f"{BIN} generate --chat --artifacts {ARTIFACTS} --vocab {VOCAB} --prompt 'aku capek banget hari ini, rasanya pengen ditemenin ngobrol' --tokens 80 --temperature 0.55 --top-k 15 --top-p 0.85 --min-p 0.05",
    shell=True,
    check=True,
)
